# 02 — Embedding Analysis
Visualize where images cluster in ResNet50's 2048-dim embedding space using PCA, t-SNE, and UMAP.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from src.embeddings import load_embeddings

X_train, y_train = load_embeddings('train')
X_val,   y_val   = load_embeddings('val')
X_test,  y_test  = load_embeddings('test')
X_all = np.concatenate([X_train, X_val, X_test])
y_all = np.concatenate([y_train, y_val, y_test])
print(f"All embeddings shape : {X_all.shape}")
print(f"Benign: {(y_all==0).sum()}  Malignant: {(y_all==1).sum()}")

## PCA — 2D Projection

In [ ]:
pca = PCA(n_components=50, random_state=42)
X_pca50 = pca.fit_transform(X_all)
X_pca2  = X_pca50[:, :2]

colors = ['#2ecc71' if y == 0 else '#e74c3c' for y in y_all]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca2[:,0], X_pca2[:,1], c=colors, alpha=0.6, s=20, edgecolors='none')
axes[0].set_title('PCA — PC1 vs PC2', fontweight='bold')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color='#2ecc71',label='Benign'),
                         Patch(color='#e74c3c',label='Malignant')])

cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, 51), cum_var, marker='o', markersize=3, color='#3498db')
axes[1].axhline(0.9, color='red', linestyle='--', label='90% variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance Ratio')
axes[1].legend()

plt.tight_layout(); plt.show()
n90 = np.argmax(cum_var >= 0.90) + 1
print(f"Components needed for 90% variance: {n90}")

## t-SNE — 2D Projection

In [ ]:
print("Running t-SNE (this takes ~30s)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca50)

plt.figure(figsize=(8, 6))
for label, name, color in [(0,'Benign','#2ecc71'), (1,'Malignant','#e74c3c')]:
    mask = y_all == label
    plt.scatter(X_tsne[mask,0], X_tsne[mask,1], c=color, label=name,
                alpha=0.6, s=25, edgecolors='none')
plt.title('t-SNE Embedding Space (ResNet50 Features)', fontsize=13, fontweight='bold')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.legend(); plt.tight_layout(); plt.show()

## UMAP — 2D Projection

In [ ]:
try:
    import umap
    print("Running UMAP...")
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_pca50)

    plt.figure(figsize=(8, 6))
    for label, name, color in [(0,'Benign','#2ecc71'), (1,'Malignant','#e74c3c')]:
        mask = y_all == label
        plt.scatter(X_umap[mask,0], X_umap[mask,1], c=color, label=name,
                    alpha=0.6, s=25, edgecolors='none')
    plt.title('UMAP Embedding Space (ResNet50 Features)', fontsize=13, fontweight='bold')
    plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    print("umap-learn not installed. Run: pip install umap-learn")

## Class Centroid Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
c0 = X_all[y_all==0].mean(axis=0)
c1 = X_all[y_all==1].mean(axis=0)
sim = cosine_similarity(c0.reshape(1,-1), c1.reshape(1,-1))[0][0]
print(f"Cosine similarity between Benign and Malignant centroids: {sim:.4f}")
print(f"L2 distance between centroids: {np.linalg.norm(c0 - c1):.4f}")
print("(Lower cosine sim / Higher L2 = more separable classes)")